# Corrective and Agentic RAG Patterns

**Naive RAG** runs retrieve → generate once. If the retriever returns wrong or empty chunks, the model still answers — grounded in garbage.

Two patterns add **feedback loops**:

| Pattern | Loop behavior | Who decides next step |
|---------|---------------|----------------------|
| **Corrective RAG (CRAG)** | Grade retrieval quality → rewrite query or fallback → re-retrieve | Fixed policy graph |
| **Agentic RAG** | Agent chooses: search again, rewrite, answer, or abstain | Planner model or rule engine |

```
Naive:     query ──► retrieve ──► answer

Corrective: query ──► retrieve ──► grade ──► (bad?) rewrite ──► retrieve ──► answer

Agentic:   query ──► agent ──► search / rewrite / answer / abstain ──► (loop)
```

This notebook implements all three on **ShopCo Policy RAG** (TF-IDF + FAISS), with relevance grading, query rewriting, iteration caps, and a golden-set comparison.

Everything is self-contained plain Python. An optional cell uses a live LLM for grading.

In [ ]:
"""
ShopCo Corrective + Agentic RAG lab.
"""

import json
import os
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Literal

import faiss
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-proj-placeholder-replace-with-your-real-key")
os.environ.setdefault("OPENAI_API_KEY", OPENAI_API_KEY)

USE_LIVE_LLM = False
MAX_RETRIEVAL_ROUNDS = 3


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


POLICY_DOCS: list[dict[str, str]] = [
    {
        "id": "pol-returns-01",
        "title": "Returns Policy",
        "text": "Customers may return items within 30 days of delivery. Items must be unused and in original packaging. Refunds in 5-7 business days. Final-sale items cannot be returned.",
    },
    {
        "id": "pol-shipping-02",
        "title": "Shipping Policy",
        "text": "Standard shipping is free on orders over $50. Express shipping at checkout. Tracking numbers emailed when the carrier scans the package.",
    },
    {
        "id": "pol-billing-03",
        "title": "Billing Policy",
        "text": "Charges appear as SHOPCO* on credit card statements. Partial refunds may take two billing cycles. Disputes require order ID.",
    },
    {
        "id": "faq-tracking-04",
        "title": "Tracking FAQ",
        "text": "Shipped orders have tracking numbers in the confirmation email. Processing orders do not yet have tracking.",
    },
    {
        "id": "faq-warranty-05",
        "title": "Warranty FAQ",
        "text": "Electronics include a 1-year manufacturer warranty. Claims require proof of purchase and serial number.",
    },
]

CHUNKS = [
    {"chunk_id": d["id"], "title": d["title"], "text": d["text"]}
    for d in POLICY_DOCS
]

print(f"Corpus: {len(CHUNKS)} chunks")

---

## 1. Naive RAG — Single-Shot Baseline

One retrieval call, one answer. No quality check on evidence.

In [ ]:
@dataclass
class RetrievalHit:
    chunk_id: str
    title: str
    text: str
    score: float


class PolicyRetriever:
    def __init__(self, chunks: list[dict[str, str]]) -> None:
        self.chunks = chunks
        texts = [f"{c['title']} {c['text']}" for c in chunks]
        self.vectorizer = TfidfVectorizer(stop_words="english")
        matrix = self.vectorizer.fit_transform(texts).astype(np.float32)
        self.vectors = matrix.toarray()
        faiss.normalize_L2(self.vectors)
        self.index = faiss.IndexFlatIP(self.vectors.shape[1])
        self.index.add(self.vectors)

    def search(self, query: str, top_k: int = 3) -> list[RetrievalHit]:
        q = self.vectorizer.transform([query]).astype(np.float32).toarray()
        faiss.normalize_L2(q)
        scores, indices = self.index.search(q, top_k)
        hits: list[RetrievalHit] = []
        for score, idx in zip(scores[0], indices[0]):
            if idx < 0:
                continue
            c = self.chunks[idx]
            hits.append(RetrievalHit(c["chunk_id"], c["title"], c["text"], float(score)))
        return hits


retriever = PolicyRetriever(CHUNKS)


def compose_answer(question: str, hits: list[RetrievalHit]) -> str:
    if not hits:
        return "No policy information found."
    top = hits[0]
    return f"[{top.chunk_id}] {top.title}: {top.text[:120]}..."


def naive_rag(question: str) -> dict[str, Any]:
    hits = retriever.search(question)
    return {
        "pattern": "naive",
        "question": question,
        "answer": compose_answer(question, hits),
        "hits": [h.chunk_id for h in hits],
        "rounds": 1,
        "trace": ["retrieve", "compose"],
    }


print(naive_rag("What is the return policy?")["answer"][:80], "...")

---

## 2. When Naive RAG Fails

Vague or mismatched queries retrieve low-relevance chunks. Naive RAG still composes an answer.

In [ ]:
TRICKY_QUERIES = [
    {"q": "I want my money back", "expect_chunk": "pol-returns-01"},
    {"q": "weird charge on card", "expect_chunk": "pol-billing-03"},
    {"q": "where is my box", "expect_chunk": "faq-tracking-04"},
    {"q": "broken laptop coverage", "expect_chunk": "faq-warranty-05"},
]

print(f"{'Query':<28} {'Naive top-1':<22} Expected")
print("-" * 65)
for row in TRICKY_QUERIES:
    hits = retriever.search(row["q"])
    top = hits[0].chunk_id if hits else "(none)"
    ok = "✓" if top == row["expect_chunk"] else "✗"
    print(f"{row['q']:<28} {top:<20} {ok}  {row['expect_chunk']}")

---

## 3. Relevance Grading — The Correction Trigger

**Corrective RAG** grades whether retrieved evidence is sufficient before composing an answer.

Graders can be:

- **Score threshold** — top hit similarity below cutoff
- **Keyword coverage** — expected domain terms missing
- **LLM judge** — "does this evidence answer the question?" (optional)

In [ ]:
@dataclass
class GradeResult:
    relevant: bool
    confidence: float
    reason: str


INTENT_KEYWORDS: dict[str, list[str]] = {
    "return": ["return", "refund", "30 days", "unused"],
    "billing": ["shopco*", "statement", "billing", "charge", "dispute"],
    "tracking": ["tracking", "shipped", "carrier", "package"],
    "warranty": ["warranty", "manufacturer", "serial", "electronics"],
    "shipping": ["shipping", "$50", "express", "free"],
}


def detect_intent(query: str) -> str:
    q = query.lower()
    if any(w in q for w in ("money back", "return", "refund")):
        return "return"
    if any(w in q for w in ("charge", "statement", "billing", "card")):
        return "billing"
    if any(w in q for w in ("box", "package", "tracking", "where is")):
        return "tracking"
    if any(w in q for w in ("warranty", "broken", "laptop", "coverage")):
        return "warranty"
    if "shipping" in q or "delivery" in q:
        return "shipping"
    return "general"


def grade_relevance(query: str, hits: list[RetrievalHit], min_score: float = 0.08) -> GradeResult:
    if not hits:
        return GradeResult(False, 0.0, "no hits")
    if hits[0].score < min_score:
        return GradeResult(False, hits[0].score, f"low vector score {hits[0].score:.3f}")

    intent = detect_intent(query)
    if intent == "general":
        return GradeResult(True, hits[0].score, "general query accepted")

    expected_terms = INTENT_KEYWORDS.get(intent, [])
    combined = " ".join(h.text.lower() for h in hits[:2])
    matches = sum(1 for t in expected_terms if t in combined)
    coverage = matches / len(expected_terms) if expected_terms else 1.0

    if coverage < 0.25:
        return GradeResult(False, coverage, f"intent={intent} term coverage {coverage:.0%}")
    return GradeResult(True, coverage, f"intent={intent} coverage ok")


sample_hits = retriever.search("I want my money back")
print(grade_relevance("I want my money back", sample_hits))

---

## 4. Query Rewriting for Corrective Retrieval

When grading fails, rewrite the query with domain vocabulary before re-searching.

In [ ]:
REWRITE_MAP: dict[str, str] = {
    "return": "ShopCo returns refund policy 30 days unused packaging",
    "billing": "ShopCo billing SHOPCO* credit card statement charge dispute",
    "tracking": "ShopCo order tracking number shipped package carrier",
    "warranty": "ShopCo electronics manufacturer warranty serial proof of purchase",
    "shipping": "ShopCo free shipping $50 express delivery",
    "general": "ShopCo customer support policy",
}


def rewrite_query(query: str, attempt: int) -> str:
    intent = detect_intent(query)
    base = REWRITE_MAP.get(intent, REWRITE_MAP["general"])
    if attempt == 1:
        return base
    return base + " " + query  # blend original on second rewrite


print("Rewrite 'money back':", rewrite_query("I want my money back", 1))

---

## 5. Corrective RAG Loop

Fixed policy graph: retrieve → grade → (if bad) rewrite → retrieve again → compose or abstain.

In [ ]:
def corrective_rag(question: str, max_rounds: int = MAX_RETRIEVAL_ROUNDS) -> dict[str, Any]:
    trace: list[str] = []
    query = question
    best_hits: list[RetrievalHit] = []

    for round_num in range(1, max_rounds + 1):
        hits = retriever.search(query)
        trace.append(f"round{round_num}:retrieve({query[:30]}...)")
        grade = grade_relevance(question, hits)
        trace.append(f"round{round_num}:grade={grade.relevant}:{grade.reason}")

        if grade.relevant:
            best_hits = hits
            break

        if round_num < max_rounds:
            query = rewrite_query(question, round_num)
            trace.append(f"round{round_num}:rewrite→{query[:40]}...")
        else:
            best_hits = hits
            trace.append("max_rounds:use_best_effort")

    final_grade = grade_relevance(question, best_hits)
    if not final_grade.relevant:
        return {
            "pattern": "corrective",
            "question": question,
            "answer": "I cannot verify a policy answer from our documents. Please contact ShopCo support.",
            "hits": [h.chunk_id for h in best_hits],
            "rounds": round_num,
            "abstained": True,
            "trace": trace,
        }

    return {
        "pattern": "corrective",
        "question": question,
        "answer": compose_answer(question, best_hits),
        "hits": [h.chunk_id for h in best_hits],
        "rounds": round_num,
        "abstained": False,
        "trace": trace,
    }


cr = corrective_rag("I want my money back")
print("Answer:", cr["answer"][:80], "...")
print("Rounds:", cr["rounds"], "| Trace:", cr["trace"])

---

## 6. Agentic RAG — Planner Chooses Actions

**Agentic RAG** gives an agent an **action space** instead of a fixed graph:

| Action | Effect |
|--------|--------|
| `search` | Run retriever with current query |
| `rewrite` | Improve query from intent map |
| `answer` | Compose from evidence if grade passes |
| `abstain` | Refuse when max steps or persistent low grade |

In [ ]:
class AgentAction(str, Enum):
    SEARCH = "search"
    REWRITE = "rewrite"
    ANSWER = "answer"
    ABSTAIN = "abstain"


@dataclass
class AgenticState:
    question: str
    query: str
    hits: list[RetrievalHit] = field(default_factory=list)
    step: int = 0
    max_steps: int = 5
    last_grade: GradeResult | None = None
    trace: list[str] = field(default_factory=list)
    done: bool = False
    answer: str = ""
    abstained: bool = False


def offline_planner(state: AgenticState) -> AgentAction:
    """Rule-based planner — stand-in for LLM agent policy."""
    if state.done:
        return AgentAction.ABSTAIN
    if state.step >= state.max_steps:
        return AgentAction.ABSTAIN
    if not state.hits:
        return AgentAction.SEARCH
    grade = grade_relevance(state.question, state.hits)
    state.last_grade = grade
    if grade.relevant:
        return AgentAction.ANSWER
    if state.step < 2:
        return AgentAction.REWRITE
    return AgentAction.SEARCH if state.step < state.max_steps - 1 else AgentAction.ABSTAIN

---

## 7. Agentic RAG Executor

In [ ]:
def execute_action(state: AgenticState, action: AgentAction) -> AgenticState:
    state.step += 1
    if action == AgentAction.SEARCH:
        state.hits = retriever.search(state.query)
        state.trace.append(f"step{state.step}:search→{len(state.hits)} hits")
    elif action == AgentAction.REWRITE:
        state.query = rewrite_query(state.question, state.step)
        state.trace.append(f"step{state.step}:rewrite→{state.query[:40]}...")
    elif action == AgentAction.ANSWER:
        state.answer = compose_answer(state.question, state.hits)
        state.done = True
        state.trace.append(f"step{state.step}:answer")
    elif action == AgentAction.ABSTAIN:
        state.abstained = True
        state.answer = "I cannot verify a policy answer. Escalating to human support."
        state.done = True
        state.trace.append(f"step{state.step}:abstain")
    return state


def agentic_rag(question: str, max_steps: int = 5) -> dict[str, Any]:
    state = AgenticState(question=question, query=question, max_steps=max_steps)
    while not state.done and state.step < max_steps:
        action = offline_planner(state)
        state = execute_action(state, action)
    if not state.done:
        state = execute_action(state, AgentAction.ABSTAIN)

    return {
        "pattern": "agentic",
        "question": question,
        "answer": state.answer,
        "hits": [h.chunk_id for h in state.hits],
        "rounds": state.step,
        "abstained": state.abstained,
        "trace": state.trace,
    }


ar = agentic_rag("weird charge on card")
print("Answer:", ar["answer"][:90], "...")
print("Trace:", ar["trace"])

---

## 8. Self-Correction — Answer Validation Loop

After composing, validate that the answer cites evidence matching the question intent. If not, trigger another search.

In [ ]:
def validate_answer(question: str, answer: str, hits: list[RetrievalHit]) -> bool:
    if not hits:
        return False
    intent = detect_intent(question)
    if intent == "general":
        return True
    top = hits[0]
    if top.chunk_id not in answer:
        return False
    terms = INTENT_KEYWORDS.get(intent, [])
    return any(t in top.text.lower() for t in terms)


def self_correcting_rag(question: str) -> dict[str, Any]:
    result = corrective_rag(question)
    if result.get("abstained"):
        return result
    chunk_map = {c["chunk_id"]: c for c in CHUNKS}
    hits = [
        RetrievalHit(cid, chunk_map[cid]["title"], chunk_map[cid]["text"], 1.0)
        for cid in result.get("hits", [])
        if cid in chunk_map
    ]
    if validate_answer(question, result["answer"], hits):
        result["trace"].append("self_check:passed")
        return result
    # one more corrective pass with rewrite
    result["trace"].append("self_check:failed→extra_round")
    extra = corrective_rag(rewrite_query(question, 1), max_rounds=2)
    extra["pattern"] = "self_correcting"
    extra["trace"] = result["trace"] + extra["trace"]
    return extra


sc = self_correcting_rag("broken laptop coverage")
print(sc["pattern"], sc["hits"], sc["trace"][-2:])

---

## 9. Pattern Comparison on Tricky Queries

In [ ]:
PATTERNS = {
    "naive": naive_rag,
    "corrective": corrective_rag,
    "agentic": agentic_rag,
}


def eval_top1(pattern_fn, query: str, expect: str) -> bool:
    out = pattern_fn(query)
    if out.get("abstained"):
        return False
    hits = out.get("hits", [])
    return bool(hits) and hits[0] == expect


print(f"{'Pattern':<12} " + " ".join(f"{row['q'][:12]:<14}" for row in TRICKY_QUERIES))
print("-" * 75)
for name, fn in PATTERNS.items():
    marks = []
    for row in TRICKY_QUERIES:
        ok = eval_top1(fn, row["q"], row["expect_chunk"])
        marks.append("✓" if ok else "✗")
    score = marks.count("✓") / len(marks)
    print(f"{name:<12} " + " ".join(f"{m:<14}" for m in marks) + f"  {score:.0%}")

---

## 10. Iteration Guards and Cost Control

Agentic and corrective loops need **hard caps**:

| Guard | Purpose |
|-------|--------|
| `max_rounds` / `max_steps` | Prevent infinite retrieve loops |
| Score plateau detection | Stop if rewrite doesn't improve grade |
| Abstain path | Escalate to human instead of guessing |
| Token budget | Limit total retrieved text per ticket |

In [ ]:
@dataclass
class LoopBudget:
    max_steps: int = 5
    max_retrieval_calls: int = 4
    max_chunks_in_context: int = 6
    retrieval_calls: int = 0

    def can_retrieve(self) -> bool:
        return self.retrieval_calls < self.max_retrieval_calls

    def record_retrieval(self) -> None:
        self.retrieval_calls += 1


def agentic_rag_with_budget(question: str, budget: LoopBudget) -> dict[str, Any]:
    state = AgenticState(question=question, query=question, max_steps=budget.max_steps)
    while not state.done and state.step < budget.max_steps:
        action = offline_planner(state)
        if action == AgentAction.SEARCH and not budget.can_retrieve():
            action = AgentAction.ABSTAIN
        if action == AgentAction.SEARCH:
            budget.record_retrieval()
        state = execute_action(state, action)
    if not state.done:
        state = execute_action(state, AgentAction.ABSTAIN)
    return {"answer": state.answer, "retrieval_calls": budget.retrieval_calls, "trace": state.trace}


b = LoopBudget(max_retrieval_calls=2)
out = agentic_rag_with_budget("where is my box", b)
print("Retrieval calls:", out["retrieval_calls"], "| Steps:", len(out["trace"]))

---

## 11. Corrective vs Agentic — When to Use Which

| Factor | Corrective RAG | Agentic RAG |
|--------|----------------|-------------|
| **Control flow** | Fixed graph | Dynamic planner |
| **Auditability** | Easier — predictable steps | Harder — log each action |
| **Flexibility** | Lower | Higher — multi-tool agents |
| **Best for** | Support bots with known failure modes | Research assistants, multi-source |
| **Testing** | Unit test grader + rewriter | Golden trajectories per action sequence |

---

## 12. Wiring into Multi-Agent Pipelines

In a support pipeline, corrective/agentic RAG typically lives inside the **retriever agent** stage:

```
  intake ──► retriever agent (corrective/agentic loop) ──► composer ──► validator
                    │
                    └── evidence bundle + trace + abstain flag
```

The validator checks `abstained` and citation IDs — it does not re-run retrieval.

In [ ]:
def retriever_agent_stage(question: str, mode: Literal["naive", "corrective", "agentic"] = "corrective") -> dict[str, Any]:
    runners = {"naive": naive_rag, "corrective": corrective_rag, "agentic": agentic_rag}
    result = runners[mode](question)
    return {
        "evidence_ids": result.get("hits", []),
        "abstained": result.get("abstained", False),
        "retrieval_trace": result.get("trace", []),
        "rounds": result.get("rounds", 1),
        "draft_context": result.get("answer", ""),
    }


handoff = retriever_agent_stage("I want my money back", mode="corrective")
print(pretty(handoff))

---

## 13. Trace Inspection — Debugging Failed Loops

In [ ]:
DEBUG_TICKETS = [
    "I want my money back",
    "quantum physics refund",
    "weird charge on card",
]

for q in DEBUG_TICKETS:
    print(f"\n=== {q} ===")
    for name, fn in PATTERNS.items():
        r = fn(q)
        print(f"  [{name}] rounds={r.get('rounds')} abstained={r.get('abstained', False)}")
        for t in r.get("trace", [])[:4]:
            print(f"    {t}")

---

## 14. Common Mistakes

| Mistake | Symptom | Fix |
|---------|---------|-----|
| No relevance grader | Wrong chunks still answered | Grade before compose |
| Infinite rewrite loop | Cost spike, timeout | `max_rounds` + abstain |
| Agentic without action log | Unreplayable failures | Trace every action |
| Corrective without abstain | Confident wrong answers | Escalate on persistent low grade |
| Re-retrieve entire pipeline | Duplicate work | Retriever agent owns the loop |

---

## 15. Optional Live LLM Grader

Set `USE_LIVE_LLM = True` to grade relevance with `gpt-4o-mini` instead of rule-based coverage.

In [ ]:
def llm_grade_relevance(query: str, hits: list[RetrievalHit]) -> GradeResult:
    from langchain_core.messages import HumanMessage, SystemMessage
    from langchain_openai import ChatOpenAI

    if not hits:
        return GradeResult(False, 0.0, "no hits")
    evidence = hits[0].text
    llm = ChatOpenAI(model="gpt-4o-mini", api_key=OPENAI_API_KEY, temperature=0)
    resp = llm.invoke([
        SystemMessage(content="Reply YES or NO only. Does the evidence answer the question?"),
        HumanMessage(content=f"Q: {query}\nEvidence: {evidence}"),
    ])
    relevant = str(resp.content).strip().upper().startswith("Y")
    return GradeResult(relevant, 0.9 if relevant else 0.1, "llm_judge")


if USE_LIVE_LLM:
    hits = retriever.search("money back")
    print(llm_grade_relevance("I want my money back", hits))
else:
    print("Offline grader:", grade_relevance("I want my money back", retriever.search("money back")))
    print("Set USE_LIVE_LLM = True for LLM relevance grading.")

---

## 16. Quiz

1. What triggers a corrective RAG rewrite pass?
2. How does agentic RAG differ from corrective RAG in control flow?
3. Why is an abstain action necessary in retrieval loops?
4. What should the retriever agent hand off to the composer?
5. Name two iteration guards that prevent runaway retrieval cost.

---

## 17. Summary

- **Naive RAG** — one shot; fails silently on bad retrieval.
- **Corrective RAG** — grade → rewrite → re-retrieve on a fixed graph; abstain when evidence never passes.
- **Agentic RAG** — planner selects `search`, `rewrite`, `answer`, or `abstain` over multiple steps.

On ShopCo tricky queries, corrective and agentic patterns recover the right policy chunk where naive RAG misses. Place the loop inside the **retriever agent**, cap iterations, log traces, and hand an evidence bundle (or abstain flag) to downstream compose/validate stages.